# Deep Neural Network using Tensorflow:

## Imports and Loading Dataset:

In [19]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import models, layers
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import HashingVectorizer

from huggingface_hub import login
from google.colab import userdata
from datasets import load_dataset

In [20]:
# Logging in to Hugging Face:
hf_token = userdata.get('HF_TOKEN')
login(token= hf_token, add_to_git_credential= True)

# loading Dataset from Hugging Face:

dataset = load_dataset('ed-donner/items_lite')

In [21]:
train, val, test = dataset['train'], dataset['validation'], dataset['test']
print(f'Train: {len(train)}')
print(f'Val: {len(val)}')
print(f'Test: {len(test)}')

Train: 20000
Val: 1000
Test: 1000


## Simple Deep Neural Network using Tensorflow:

In [22]:
# Documents (item Summary) and Prices:
y = np.array([float(item['price']) for item in train])
documents = [item['summary'] for item in train]

In [23]:
# Using the HashingVectorizer for a Bag of Words Model
# Using binary=True with the CountVectorizer makes "one-hot vectors"

np.random.seed(42)
vectorizer = HashingVectorizer(n_features= 5000, stop_words='english', binary=True)
X = vectorizer.fit_transform(documents)

In [24]:
# Converting Sparse Matrix to Dense:
X = X.toarray()

In [25]:
# Train Test Split:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size= 0.1, random_state= 42)

### Preparig Data:

In [26]:
train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train))
val_dataset = tf.data.Dataset.from_tensor_slices((X_val, y_val))

train_dataset = train_dataset.shuffle(buffer_size= 1024).batch(32).prefetch(tf.data.AUTOTUNE)
val_dataset = val_dataset.batch(64)

### Defining Neural Network using Tensorflow:

In [27]:
# Creating The Model:
model = models.Sequential()
model.add(layers.Dense(128, activation='relu', input_shape= (X_train.shape[1],)))
model.add(layers.Dense(64, activation='relu'))
model.add(layers.Dense(64, activation='relu'))
model.add(layers.Dense(64, activation='relu'))
model.add(layers.Dense(64, activation='relu'))
model.add(layers.Dense(64, activation='relu'))
model.add(layers.Dense(64, activation='relu'))
model.add(layers.Dense(1))

# Compiling The Model:
model.compile(optimizer= tf.keras.optimizers.Adam(learning_rate= 0.001),
              loss= 'mse')

# Training The Model:
history = model.fit(
    train_dataset,
    validation_data = val_dataset,
    epochs= 2
)

Epoch 1/2


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


563/563 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step - loss: 20387.2070 - val_loss: 19302.3105
Epoch 2/2
563/563 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 13523.8320 - val_loss: 18399.3457
